<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install
!pip install pandas -q

In [ ]:
# Cell 2: Upload dataset.jsonl จากคอม
from google.colab import files
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f'Uploaded: {INPUT_FILENAME}')

In [ ]:
# Cell 3: Import + Config
import json, re, os, random
import pandas as pd
from collections import defaultdict, Counter

INPUT_JSONL  = INPUT_FILENAME
OUT_DIR      = '/content/clean_output'
REVIEW_CSV   = os.path.join(OUT_DIR, 'dataset_review.csv')
EDITED_CSV   = os.path.join(OUT_DIR, 'dataset_review_edited.csv')
OUTPUT_JSONL = os.path.join(OUT_DIR, 'dataset_cleaned.jsonl')
SAMPLE_N     = 20

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Cell 4: Load dataset
data = []
with open(INPUT_JSONL, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        item = json.loads(line.strip())
        item['id'] = i
        data.append(item)

print(f'Total: {len(data)}')
print(f'Label 0: {sum(1 for d in data if d["label"]==0)}')
print(f'Label 1: {sum(1 for d in data if d["label"]==1)}')

In [ ]:
# Cell 5: สรุป category / subtype
print('Category:')
for k, v in Counter(d['category'] for d in data).most_common():
    print(f'  {k}: {v}')

print('\nSubtype:')
for k, v in Counter(d['subtype'] for d in data).most_common():
    print(f'  {k}: {v}')

In [ ]:
# Cell 6: Inspect samples per subtype (20 ตัว/subtype)
by_subtype = defaultdict(list)
for d in data:
    by_subtype[d['subtype']].append(d)

random.seed(42)
for subtype in sorted(by_subtype.keys()):
    items = by_subtype[subtype]
    samples = random.sample(items, min(SAMPLE_N, len(items)))
    print(f'\n=== {subtype} (label={samples[0]["label"]}, n={len(items)}) ===')
    for item in samples:
        print(f'\nid={item["id"]}')
        for t in item['turns']:
            print(f'  {t["speaker"]}: {t["text"]}')

In [ ]:
# Cell 7: Remove unwanted ids / subtypes
remove_ids = []
remove_subtypes = []

data = [d for d in data
        if d['id'] not in set(remove_ids)
        and d['subtype'] not in set(remove_subtypes)]

print(f'Remaining: {len(data)}')

In [1]:
# Cell 8: Clean text (ลบ ! ? ... … emoji + แทน URL ทั้งหมดด้วย "ลิงก์") + ลบ duplicate
URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+|bit\.ly/\S+)', re.IGNORECASE)
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F600-\U0001F64F'
    '\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF'
    '\U0001F1E0-\U0001F1FF'
    '\U00002500-\U00002BEF'
    '\U00002702-\U000027B0'
    '\U0001F900-\U0001F9FF'
    '\U0001FA00-\U0001FA6F'
    '\U0001FA70-\U0001FAFF'
    ']+', flags=re.UNICODE
)

def clean_text(text):
    # แทน URL ทั้งหมดด้วยคำว่า "ลิงก์" (เพื่อให้ TTS อ่านออกเสียงได้)
    text = URL_PATTERN.sub('ลิงก์', text)
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'[!?]+', '', text)
    text = re.sub(r'\.{2,}', ' ', text)
    text = text.replace('…', ' ')
    text = re.sub(r'[~^*<>{}\[\]\\|`]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for item in data:
    for turn in item['turns']:
        turn['text'] = clean_text(turn['text'])

# ลบ duplicate
seen = set()
deduped = []
for d in data:
    key = tuple((t['speaker'], t['text']) for t in d['turns'])
    if key not in seen:
        seen.add(key)
        deduped.append(d)

print(f'Before dedup: {len(data)}')
print(f'After dedup : {len(deduped)}')
print(f'Removed dup : {len(data) - len(deduped)}')
data = deduped

NameError: name 're' is not defined

In [ ]:
# Cell 9: Export CSV เพื่อ review
import base64
from IPython.display import HTML

rows = []
for d in data:
    full_text = ' | '.join(f'[{t["speaker"]}] {t["text"]}' for t in d['turns'])
    rows.append({
        'id': d['id'],
        'label': d['label'],
        'category': d['category'],
        'subtype': d['subtype'],
        'num_turns': len(d['turns']),
        'full_text': full_text,
        'remove': '',
    })

review_df = pd.DataFrame(rows)
review_df.to_csv(REVIEW_CSV, index=False, encoding='utf-8-sig')
print(f'Saved: {REVIEW_CSV}')

# สร้าง download link
with open(REVIEW_CSV, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f'<a href="data:text/csv;base64,{b64}" download="dataset_review.csv">คลิกที่นี่เพื่อ Download CSV</a>'))

In [ ]:
# Cell 10: Upload CSV ที่แก้แล้วกลับมา (เปิดใน Excel ใส่ x ใน column remove)
from google.colab import files
uploaded = files.upload()
edited_filename = list(uploaded.keys())[0]

edited_df = pd.read_csv(edited_filename, encoding='utf-8-sig')
edited_df['remove'] = edited_df['remove'].fillna('').astype(str).str.strip().str.lower()
keep_df = edited_df[edited_df['remove'] != 'x']

print(f'Removed: {len(edited_df) - len(keep_df)}')
print(f'Remaining: {len(keep_df)}')

In [ ]:
# Cell 11: Export JSONL พร้อมไปเจนเสียง
keep_ids = set(keep_df['id'].tolist())
cleaned_data = [d for d in data if d['id'] in keep_ids]

with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for d in cleaned_data:
        out = {'label': d['label'], 'turns': d['turns']}
        f.write(json.dumps(out, ensure_ascii=False) + '\n')

print(f'Saved: {OUTPUT_JSONL}')
print(f'Total: {len(cleaned_data)}')

# สร้าง download link
with open(OUTPUT_JSONL, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f'<a href="data:application/json;base64,{b64}" download="dataset_cleaned.jsonl">คลิกที่นี่เพื่อ Download JSONL</a>'))